# 06. Desarrollo y selección de modelos de clasificación

Este cuaderno compara modelos de control y modelos con información cinemática utilizando únicamente 2019–2024. La validación interna es temporal creciente y el resultado se utiliza para seleccionar y congelar los modelos que después serán evaluados en 2025.


In [ ]:
import hashlib
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    PolynomialFeatures,
    StandardScaler,
)


## 1. Carga y contrato de entrada


### Base de entrenamiento y metadatos


In [ ]:
# Localizar la raíz y comprobar únicamente los productos de desarrollo

ruta_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        ruta
        for ruta in [ruta_actual, *ruta_actual.parents]
        if (ruta / "pyproject.toml").is_file()
    ),
    None,
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code o Jupyter desde el repositorio."
    )

carpeta_modelado = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_modelado_v2"
)

ruta_entrenamiento = (
    carpeta_modelado
    / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
)

ruta_diccionario = (
    carpeta_modelado
    / "metadata"
    / "diccionario_variables_modelado_v2.csv"
)

ruta_configuracion = (
    carpeta_modelado
    / "metadata"
    / "configuracion_modelado_v2.csv"
)

rutas_entrada = {
    "Entrenamiento": ruta_entrenamiento,
    "Diccionario": ruta_diccionario,
    "Configuración": ruta_configuracion,
}

archivos_faltantes = [
    str(ruta.relative_to(raiz_proyecto))
    for ruta in rutas_entrada.values()
    if not ruta.is_file()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan productos del notebook 04:\n"
        + "\n".join(archivos_faltantes)
    )

display(
    pd.DataFrame(
        {
            "archivo": rutas_entrada.keys(),
            "ruta": [
                str(ruta.relative_to(raiz_proyecto))
                for ruta in rutas_entrada.values()
            ],
        }
    )
)


In [ ]:
# Cargar entrenamiento y comprobar su contrato

df_entrenamiento = pd.read_parquet(ruta_entrenamiento)
df_diccionario = pd.read_csv(ruta_diccionario)
df_configuracion = pd.read_csv(ruta_configuracion)

variables_documentadas = set(df_diccionario["variable"])
columnas_entrenamiento = set(df_entrenamiento.columns)

comprobaciones_estructura = pd.DataFrame(
    [
        {
            "comprobacion": "Todas las columnas están documentadas",
            "resultado": columnas_entrenamiento == variables_documentadas,
        },
        {
            "comprobacion": "Identificadores únicos",
            "resultado": df_entrenamiento["id_fila_origen"].is_unique,
        },
        {
            "comprobacion": "Entrenamiento contiene 2019–2024",
            "resultado": (
                set(df_entrenamiento["source_year"].unique())
                == set(range(2019, 2025))
            ),
        },
        {
            "comprobacion": "Partición de entrenamiento correcta",
            "resultado": (
                set(df_entrenamiento["particion_modelado"].unique())
                == {"entrenamiento"}
            ),
        },
    ]
)

resumen_base = pd.DataFrame(
    [
        {
            "filas": len(df_entrenamiento),
            "columnas": df_entrenamiento.shape[1],
            "anio_minimo": int(df_entrenamiento["source_year"].min()),
            "anio_maximo": int(df_entrenamiento["source_year"].max()),
        }
    ]
)

display(resumen_base)
display(comprobaciones_estructura)

if not comprobaciones_estructura["resultado"].all():
    raise ValueError(
        "La base de entrenamiento no satisface el contrato esperado."
    )


### Muestras utilizables para clasificación


In [ ]:
# Construir las muestras estándar y estricta utilizables para clasificación

columnas_seleccion_ml = [
    "id_fila_origen",
    "usar_modelo_ml",
    "objetivo_ml",
    "grupo_brillo_ml",
    "pertenece_muestra_estricta",
]

columnas_ausentes_ml = [
    columna
    for columna in columnas_seleccion_ml
    if columna not in df_entrenamiento.columns
]

if columnas_ausentes_ml:
    raise KeyError(
        "Faltan columnas para construir las muestras de ML: "
        f"{columnas_ausentes_ml}"
    )

df_entrenamiento_ml = df_entrenamiento.loc[
    df_entrenamiento["usar_modelo_ml"]
].copy()

df_entrenamiento_ml_estricto = df_entrenamiento_ml.loc[
    df_entrenamiento_ml["pertenece_muestra_estricta"]
].copy()

resumen_muestras_ml = pd.DataFrame(
    [
        {
            "muestra": "Estándar",
            "trayectorias_base": len(df_entrenamiento),
            "trayectorias_utilizables_ml": len(df_entrenamiento_ml),
            "porcentaje_utilizable": (
                100 * len(df_entrenamiento_ml) / len(df_entrenamiento)
            ),
        },
        {
            "muestra": "Estricta",
            "trayectorias_base": int(
                df_entrenamiento[
                    "pertenece_muestra_estricta"
                ].sum()
            ),
            "trayectorias_utilizables_ml": len(
                df_entrenamiento_ml_estricto
            ),
            "porcentaje_utilizable": (
                100
                * len(df_entrenamiento_ml_estricto)
                / df_entrenamiento[
                    "pertenece_muestra_estricta"
                ].sum()
            ),
        },
    ]
)

comprobaciones_muestras_ml = pd.DataFrame(
    [
        {
            "comprobacion": "Objetivo completo",
            "resultado": df_entrenamiento_ml[
                "objetivo_ml"
            ].notna().all(),
        },
        {
            "comprobacion": "Objetivo binario",
            "resultado": (
                set(df_entrenamiento_ml["objetivo_ml"].unique())
                == {0, 1}
            ),
        },
        {
            "comprobacion": "IDs únicos",
            "resultado": df_entrenamiento_ml[
                "id_fila_origen"
            ].is_unique,
        },
        {
            "comprobacion": "Estricta es subconjunto de estándar",
            "resultado": set(
                df_entrenamiento_ml_estricto["id_fila_origen"]
            ).issubset(
                set(df_entrenamiento_ml["id_fila_origen"])
            ),
        },
    ]
)

display(
    resumen_muestras_ml.round(
        {"porcentaje_utilizable": 2}
    )
)
display(comprobaciones_muestras_ml)

if not comprobaciones_muestras_ml["resultado"].all():
    raise ValueError(
        "La construcción de las muestras detectó una inconsistencia."
    )


### Objetivo binario


In [ ]:
# Auditar la codificación, los límites y el balance del objetivo binario de entrenamiento

configuracion_modelado = df_configuracion.set_index("elemento")["valor"]
limite_brillante = float(configuracion_modelado.loc["Límite brillante"])
limite_tenue = float(configuracion_modelado.loc["Límite tenue"])

y_entrenamiento = df_entrenamiento_ml["objetivo_ml"].astype(int)
y_entrenamiento_estricto = df_entrenamiento_ml_estricto["objetivo_ml"].astype(int)

resumen_objetivo = (
    df_entrenamiento_ml.groupby("objetivo_ml", observed=True)
    .agg(
        trayectorias=("id_fila_origen", "size"),
        magnitud_minima=("magnitud_analisis", "min"),
        magnitud_mediana=("magnitud_analisis", "median"),
        magnitud_maxima=("magnitud_analisis", "max"),
    )
    .reset_index()
)

resumen_objetivo["clase"] = resumen_objetivo["objetivo_ml"].map({0: "Brillante", 1: "Tenue"})
resumen_objetivo["porcentaje_entrenamiento"] = 100 * resumen_objetivo["trayectorias"] / len(df_entrenamiento_ml)
resumen_objetivo = resumen_objetivo[["objetivo_ml", "clase", "trayectorias", "porcentaje_entrenamiento", "magnitud_minima", "magnitud_mediana", "magnitud_maxima"]]

conteos_estandar = y_entrenamiento.value_counts()
conteos_estrictos = y_entrenamiento_estricto.value_counts()

comprobaciones_objetivo = pd.DataFrame(
    [
        {"comprobacion": "Codificación binaria completa", "resultado": set(y_entrenamiento.unique()) == {0, 1}},
        {"comprobacion": "Clase brillante respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 0, "magnitud_analisis"].le(limite_brillante).all()},
        {"comprobacion": "Clase tenue respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 1, "magnitud_analisis"].ge(limite_tenue).all()},
        {"comprobacion": "Estándar contiene 4,300 brillantes", "resultado": conteos_estandar.get(0, 0) == 4300},
        {"comprobacion": "Estándar contiene 4,332 tenues", "resultado": conteos_estandar.get(1, 0) == 4332},
        {"comprobacion": "Estricta contiene 1,681 brillantes", "resultado": conteos_estrictos.get(0, 0) == 1681},
        {"comprobacion": "Estricta contiene 1,046 tenues", "resultado": conteos_estrictos.get(1, 0) == 1046},
    ]
)

print(f"Límite brillante: {limite_brillante:.2f}")
print(f"Límite tenue: {limite_tenue:.2f}")
display(resumen_objetivo.round(4))
display(comprobaciones_objetivo)

if not comprobaciones_objetivo["resultado"].all():
    raise ValueError("La auditoría del objetivo binario detectó alguna inconsistencia.")

### Bloques de predictores


In [ ]:
# Definir los bloques de predictores y comprobar su integridad numérica

variables_fase = ["delta_sol_lon_deg"]

variables_calidad = [
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "numero_estaciones",
    "sigma_radiante_aprox",
    "Vgeo_sigma_modelo",
]

variables_cinematica = [
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
]

variables_validacion = ["source_year"]

bloques_predictores = {
    "Fase solar": variables_fase,
    "Calidad observacional": variables_calidad,
    "Cinemática residual local": variables_cinematica,
}

variables_requeridas = (
    variables_fase
    + variables_calidad
    + variables_cinematica
    + variables_validacion
)

variables_ausentes = [
    variable
    for variable in variables_requeridas
    if variable not in df_entrenamiento_ml.columns
]

if variables_ausentes:
    raise KeyError(
        f"Faltan variables necesarias para el modelado: {variables_ausentes}"
    )

filas_resumen_entrenamiento = []

for bloque, variables in bloques_predictores.items():
    for variable in variables:
        valores = pd.to_numeric(
            df_entrenamiento_ml[variable],
            errors="coerce",
        )

        filas_resumen_entrenamiento.append(
            {
                "bloque": bloque,
                "variable": variable,
                "faltantes": int(valores.isna().sum()),
                "no_finitos": int(
                    (~np.isfinite(valores.dropna())).sum()
                ),
                "minimo": valores.min(),
                "mediana": valores.median(),
                "maximo": valores.max(),
            }
        )

resumen_predictores_entrenamiento = pd.DataFrame(
    filas_resumen_entrenamiento
)

display(resumen_predictores_entrenamiento.round(6))

if (
    resumen_predictores_entrenamiento[
        ["faltantes", "no_finitos"]
    ].to_numpy().sum()
    != 0
):
    raise ValueError(
        "Los predictores de entrenamiento contienen "
        "valores faltantes o no finitos."
    )


## 2. Validación temporal interna


In [ ]:
# Construir y auditar los pliegues de validación temporal creciente
# Esta validación es condicional: el centro físico y los cortes de clase
# fueron fijados por el notebook 04 con todo 2019–2024. Dentro de cada
# pliegue se ajustan únicamente con años anteriores los escaladores,
# el PCA correspondiente y la regresión logística.

def construir_pliegues_temporales(df_modelado, nombre_muestra):
    anios = sorted(df_modelado["source_year"].unique())
    pliegues = []
    filas_resumen = []

    for anio_validacion in anios[1:]:
        mascara_entrenamiento = df_modelado["source_year"] < anio_validacion
        mascara_validacion = df_modelado["source_year"] == anio_validacion
        indices_entrenamiento = np.flatnonzero(mascara_entrenamiento.to_numpy())
        indices_validacion = np.flatnonzero(mascara_validacion.to_numpy())
        objetivo_entrenamiento = df_modelado.iloc[indices_entrenamiento]["objetivo_ml"].astype(int)
        objetivo_validacion = df_modelado.iloc[indices_validacion]["objetivo_ml"].astype(int)

        pliegues.append((indices_entrenamiento, indices_validacion))
        filas_resumen.append(
            {
                "muestra": nombre_muestra,
                "pliegue": len(pliegues),
                "anios_entrenamiento": f"{df_modelado.iloc[indices_entrenamiento]['source_year'].min()}–{df_modelado.iloc[indices_entrenamiento]['source_year'].max()}",
                "anio_validacion": anio_validacion,
                "entrenamiento_total": len(indices_entrenamiento),
                "entrenamiento_brillantes": (objetivo_entrenamiento == 0).sum(),
                "entrenamiento_tenues": (objetivo_entrenamiento == 1).sum(),
                "validacion_total": len(indices_validacion),
                "validacion_brillantes": (objetivo_validacion == 0).sum(),
                "validacion_tenues": (objetivo_validacion == 1).sum(),
            }
        )

    return pliegues, pd.DataFrame(filas_resumen)

pliegues_temporales_estandar, resumen_pliegues_estandar = construir_pliegues_temporales(df_entrenamiento_ml, "Estándar")
pliegues_temporales_estrictos, resumen_pliegues_estrictos = construir_pliegues_temporales(df_entrenamiento_ml_estricto, "Estricta")
resumen_pliegues_temporales = pd.concat([resumen_pliegues_estandar, resumen_pliegues_estrictos], ignore_index=True)

comprobaciones_pliegues = pd.DataFrame(
    [
        {"comprobacion": "Cinco pliegues estándar", "resultado": len(pliegues_temporales_estandar) == 5},
        {"comprobacion": "Cinco pliegues estrictos", "resultado": len(pliegues_temporales_estrictos) == 5},
        {"comprobacion": "Ambas clases presentes en cada entrenamiento", "resultado": (resumen_pliegues_temporales[["entrenamiento_brillantes", "entrenamiento_tenues"]] > 0).all().all()},
        {"comprobacion": "Ambas clases presentes en cada validación", "resultado": (resumen_pliegues_temporales[["validacion_brillantes", "validacion_tenues"]] > 0).all().all()},
        {"comprobacion": "2025 permanece fuera de la validación interna", "resultado": resumen_pliegues_temporales["anio_validacion"].max() == 2024},
    ]
)

display(resumen_pliegues_temporales)
display(comprobaciones_pliegues)

if not comprobaciones_pliegues["resultado"].all():
    raise ValueError("El esquema de validación temporal no satisface todas las condiciones.")


### Preprocesamiento dentro de cada pliegue


In [ ]:
# Definir el preprocesamiento de fase, calidad y cinemática sin ajustarlo todavía

transformador_fase = Pipeline(
    [
        ("terminos_fase", PolynomialFeatures(degree=2, include_bias=False)),
        ("escalamiento_fase", StandardScaler()),
    ]
)

transformador_calidad = Pipeline(
    [
        ("escalamiento_calidad", StandardScaler()),
    ]
)

transformador_cinematica = Pipeline(
    [
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocesador_control_cinematica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

configuracion_preprocesamiento = pd.DataFrame(
    [
        {
            "bloque": "Fase solar",
            "variables_originales": ", ".join(variables_fase),
            "transformacion": "Término lineal + término cuadrático + estandarización",
            "numero_salidas": 2,
        },
        {
            "bloque": "Calidad observacional",
            "variables_originales": ", ".join(variables_calidad),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_calidad),
        },
        {
            "bloque": "Cinemática residual local",
            "variables_originales": ", ".join(variables_cinematica),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_cinematica),
        },
    ]
)

configuracion_modelos_iniciales = pd.DataFrame(
    [
        {
            "modelo": "Control",
            "fase": True,
            "calidad": True,
            "cinematica": False,
            "numero_predictores_transformados": 2 + len(variables_calidad),
        },
        {
            "modelo": "Control + cinemática local",
            "fase": True,
            "calidad": True,
            "cinematica": True,
            "numero_predictores_transformados": 2 + len(variables_calidad) + len(variables_cinematica),
        },
    ]
)

display(configuracion_preprocesamiento)
display(configuracion_modelos_iniciales)

if hasattr(preprocesador_control, "transformers_") or hasattr(preprocesador_control_cinematica, "transformers_"):
    raise RuntimeError("Los preprocesadores no deben estar ajustados antes de la validación temporal.")

## 3. Modelos candidatos


### Modelo de control


In [ ]:
# Entrenar y evaluar temporalmente el modelo de control mediante regresión logística

def evaluar_modelo_temporal(modelo, nombre_modelo, df_modelado, pliegues, preparar_pliegue=None):
    filas_metricas = []
    tablas_predicciones = []

    for numero_pliegue, (indices_entrenamiento, indices_validacion) in enumerate(pliegues, start=1):
        modelo_pliegue = clone(modelo)
        X_entrenamiento = df_modelado.iloc[indices_entrenamiento]
        y_entrenamiento_pliegue = X_entrenamiento["objetivo_ml"].astype(int)
        X_validacion = df_modelado.iloc[indices_validacion]
        y_validacion = X_validacion["objetivo_ml"].astype(int)
        anio_validacion = int(X_validacion["source_year"].iloc[0])
        if preparar_pliegue is not None:
            X_entrenamiento, X_validacion = preparar_pliegue(
                X_entrenamiento.copy(),
                X_validacion.copy(),
                anio_validacion,
            )

        modelo_pliegue.fit(X_entrenamiento, y_entrenamiento_pliegue)

        probabilidades = modelo_pliegue.predict_proba(X_validacion)[:, 1]
        predicciones = (probabilidades >= 0.5).astype(int)

        tabla_predicciones = pd.DataFrame(
            {
                "id_fila_origen": X_validacion["id_fila_origen"].to_numpy(),
                "modelo": nombre_modelo,
                "pliegue": numero_pliegue,
                "anio_validacion": anio_validacion,
                "objetivo_real": y_validacion.to_numpy(),
                "probabilidad_tenue": probabilidades,
                "prediccion": predicciones,
                "pertenece_muestra_estricta": X_validacion["pertenece_muestra_estricta"].to_numpy(),
            }
        )

        tablas_predicciones.append(tabla_predicciones)

        for nombre_muestra, mascara in [
            ("Estándar", np.ones(len(X_validacion), dtype=bool)),
            ("Estricta", X_validacion["pertenece_muestra_estricta"].to_numpy(dtype=bool)),
        ]:
            y_real_muestra = y_validacion.to_numpy()[mascara]
            probabilidades_muestra = probabilidades[mascara]
            predicciones_muestra = predicciones[mascara]
            verdaderos_brillantes, falsos_tenues, falsos_brillantes, verdaderos_tenues = confusion_matrix(y_real_muestra, predicciones_muestra, labels=[0, 1]).ravel()

            filas_metricas.append(
                {
                    "modelo": nombre_modelo,
                    "muestra": nombre_muestra,
                    "pliegue": numero_pliegue,
                    "anio_validacion": anio_validacion,
                    "trayectorias": len(y_real_muestra),
                    "brillantes": (y_real_muestra == 0).sum(),
                    "tenues": (y_real_muestra == 1).sum(),
                    "roc_auc": roc_auc_score(y_real_muestra, probabilidades_muestra),
                    "average_precision": average_precision_score(y_real_muestra, probabilidades_muestra),
                    "exactitud": accuracy_score(y_real_muestra, predicciones_muestra),
                    "exactitud_balanceada": balanced_accuracy_score(y_real_muestra, predicciones_muestra),
                    "sensibilidad_tenues": recall_score(y_real_muestra, predicciones_muestra, pos_label=1),
                    "especificidad_brillantes": recall_score(y_real_muestra, predicciones_muestra, pos_label=0),
                    "verdaderos_brillantes": verdaderos_brillantes,
                    "falsos_tenues": falsos_tenues,
                    "falsos_brillantes": falsos_brillantes,
                    "verdaderos_tenues": verdaderos_tenues,
                }
            )

    return pd.DataFrame(filas_metricas), pd.concat(tablas_predicciones, ignore_index=True)

modelo_control = Pipeline(
    [
        ("preprocesamiento", preprocesador_control),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_control, predicciones_control = evaluar_modelo_temporal(modelo_control, "Control", df_entrenamiento_ml, pliegues_temporales_estandar)

columnas_metricas = ["roc_auc", "average_precision", "exactitud", "exactitud_balanceada", "sensibilidad_tenues", "especificidad_brillantes"]
resumen_control = metricas_control.groupby(["modelo", "muestra"])[columnas_metricas].agg(["mean", "std"]).reset_index()
resumen_control.columns = ["modelo", "muestra"] + [f"{metrica}_{estadistico}" for metrica, estadistico in resumen_control.columns.tolist()[2:]]

display(metricas_control.round(4))
display(resumen_control.round(4))

if len(predicciones_control) != sum(len(indices_validacion) for _, indices_validacion in pliegues_temporales_estandar):
    raise ValueError("No se conservaron todas las predicciones temporales esperadas.")

### Cinemática local lineal


#### Modelo lineal


In [ ]:
# Entrenar el modelo cinemático lineal y comparar su desempeño con el modelo de control

modelo_cinematica_lineal = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_lineal, predicciones_cinematica_lineal = evaluar_modelo_temporal(
    modelo_cinematica_lineal,
    "Control + cinemática lineal",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

comparacion_control_cinematica = metricas_control.merge(
    metricas_cinematica_lineal,
    on=["muestra", "pliegue", "anio_validacion"],
    suffixes=("_control", "_cinematica"),
    validate="one_to_one",
)

for metrica in columnas_metricas:
    comparacion_control_cinematica[f"delta_{metrica}"] = comparacion_control_cinematica[f"{metrica}_cinematica"] - comparacion_control_cinematica[f"{metrica}_control"]

filas_resumen_mejoras = []

for muestra, grupo in comparacion_control_cinematica.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in columnas_metricas:
        diferencias = grupo[f"delta_{metrica}"]
        fila[f"delta_medio_{metrica}"] = diferencias.mean()
        fila[f"desviacion_delta_{metrica}"] = diferencias.std()
        fila[f"pliegues_con_mejora_{metrica}"] = (diferencias > 0).sum()

    filas_resumen_mejoras.append(fila)

resumen_mejoras_cinematica = pd.DataFrame(filas_resumen_mejoras)
columnas_comparacion_anual = ["muestra", "anio_validacion"] + [f"delta_{metrica}" for metrica in columnas_metricas]

display(metricas_cinematica_lineal.round(4))
display(comparacion_control_cinematica[columnas_comparacion_anual].round(4))
display(resumen_mejoras_cinematica.round(4))

if len(metricas_cinematica_lineal) != len(metricas_control):
    raise ValueError("El modelo cinemático y el modelo de control no produjeron el mismo número de evaluaciones.")

#### Modelo cuadrático diagonal


In [ ]:
# Entrenar el modelo cinemático cuadrático y compararlo con los modelos anteriores
sys.path.insert(0, str(raiz_proyecto))
from src.geminidas_transformaciones import agregar_cuadrados_cinematicos

transformador_cinematica_cuadratica = Pipeline(
    [
        ("terminos_cinematicos", FunctionTransformer(agregar_cuadrados_cinematicos, validate=False)),
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control_cinematica_cuadratica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica_cuadratica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

modelo_cinematica_cuadratica = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica_cuadratica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_cuadratica, predicciones_cinematica_cuadratica = evaluar_modelo_temporal(
    modelo_cinematica_cuadratica,
    "Control + cinemática cuadrática",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

claves_comparacion = ["muestra", "pliegue", "anio_validacion"]
metricas_clave = ["roc_auc", "average_precision", "exactitud_balanceada"]

comparacion_tres_modelos = (
    metricas_control[claves_comparacion + metricas_clave]
    .rename(columns={metrica: f"{metrica}_control" for metrica in metricas_clave})
    .merge(
        metricas_cinematica_lineal[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_lineal" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
    .merge(
        metricas_cinematica_cuadratica[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_cuadratica" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
)

for metrica in metricas_clave:
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_control"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_control"]
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_lineal"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_lineal"]

filas_resumen_cuadratico = []

for muestra, grupo in comparacion_tres_modelos.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in metricas_clave:
        diferencia_control = grupo[f"delta_{metrica}_cuadratica_vs_control"]
        diferencia_lineal = grupo[f"delta_{metrica}_cuadratica_vs_lineal"]
        fila[f"delta_medio_{metrica}_vs_control"] = diferencia_control.mean()
        fila[f"pliegues_mejora_{metrica}_vs_control"] = (diferencia_control > 0).sum()
        fila[f"delta_medio_{metrica}_vs_lineal"] = diferencia_lineal.mean()
        fila[f"pliegues_mejora_{metrica}_vs_lineal"] = (diferencia_lineal > 0).sum()

    filas_resumen_cuadratico.append(fila)

resumen_modelo_cuadratico = pd.DataFrame(filas_resumen_cuadratico)

display(metricas_cinematica_cuadratica.round(4))
display(comparacion_tres_modelos.round(4))
display(resumen_modelo_cuadratico.round(4))

if len(metricas_cinematica_cuadratica) != len(metricas_control):
    raise ValueError("El modelo cuadrático no produjo el número esperado de evaluaciones.")


### Modelos con PCA


#### PCA lineal con dos componentes


In [ ]:
def preparar_pca_pliegue(X_entrenamiento, X_validacion, anio_validacion):

    # El PCA utiliza todos los meteoros elegibles de años anteriores
    mascara_pca = (
        df_entrenamiento["usar_pca"]
        & (df_entrenamiento["source_year"] < anio_validacion)
    )

    datos_pca = df_entrenamiento.loc[mascara_pca, variables_cinematica]

    pca_pliegue = PCA(n_components=3)
    pca_pliegue.fit(datos_pca)

    # Proyectar entrenamiento y validación con los mismos ejes
    Z_entrenamiento = pca_pliegue.transform(
        X_entrenamiento[variables_cinematica]
    )

    Z_validacion = pca_pliegue.transform(
        X_validacion[variables_cinematica]
    )

    for i in range(3):
        columna = f"PC{i+1}"
        X_entrenamiento[columna] = Z_entrenamiento[:, i]
        X_validacion[columna] = Z_validacion[:, i]

    return X_entrenamiento, X_validacion


# Fase y calidad conservan el mismo preprocesamiento que el modelo de control
preprocesador_pca_2 = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        ("pca", StandardScaler(), ["PC1", "PC2"]),
    ],
    remainder="drop",
)


modelo_pca_2 = Pipeline(
    [
        ("preprocesamiento", preprocesador_pca_2),
        (
            "clasificador",
            LogisticRegression(
                C=1.0,
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
            ),
        ),
    ]
)


metricas_pca_2, predicciones_pca_2 = evaluar_modelo_temporal(
    modelo_pca_2,
    "Control + PCA lineal (2 PC)",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
    preparar_pliegue=preparar_pca_pliegue,
)

display(metricas_pca_2.round(4))


In [ ]:
comparacion_pca_2 = (
    metricas_pca_2.loc[
        metricas_pca_2["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc", "average_precision"]
    ]
    .merge(
        metricas_control.loc[
            metricas_control["muestra"] == "Estándar",
            ["anio_validacion", "roc_auc", "average_precision"]
        ],
        on="anio_validacion",
        suffixes=("_pca", "_control"),
        validate="one_to_one",
    )
)

comparacion_pca_2["mejora_roc_auc"] = (
    comparacion_pca_2["roc_auc_pca"]
    - comparacion_pca_2["roc_auc_control"]
)

display(comparacion_pca_2.round(4))

print(
    "Mejora media de ROC-AUC:",
    round(comparacion_pca_2["mejora_roc_auc"].mean(), 4)
)

#### PCA lineal con tres componentes


In [ ]:
# Modelo lineal con las tres componentes principales

preprocesador_pca_3 = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        ("pca", StandardScaler(), ["PC1", "PC2", "PC3"]),
    ],
    remainder="drop",
)

modelo_pca_3 = Pipeline(
    [
        ("preprocesamiento", preprocesador_pca_3),
        (
            "clasificador",
            LogisticRegression(
                C=1.0,
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
            ),
        ),
    ]
)

metricas_pca_3, predicciones_pca_3 = evaluar_modelo_temporal(
    modelo_pca_3,
    "Control + PCA lineal (3 PC)",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
    preparar_pliegue=preparar_pca_pliegue,
)

display(
    metricas_pca_3[
        ["muestra", "anio_validacion", "roc_auc", "average_precision",
         "exactitud_balanceada"]
    ].round(4)
)

In [ ]:
# Comparar los modelos sobre los mismos años

modelos_lineales = {
    "Control": metricas_control,
    "Local_3": metricas_cinematica_lineal,
    "PCA_2": metricas_pca_2,
    "PCA_3": metricas_pca_3,
}

tabla_lineales = None

for nombre, metricas in modelos_lineales.items():
    datos = metricas.loc[
        metricas["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc"]
    ].rename(columns={"roc_auc": nombre})

    if tabla_lineales is None:
        tabla_lineales = datos
    else:
        tabla_lineales = tabla_lineales.merge(
            datos,
            on="anio_validacion",
            validate="one_to_one"
        )

tabla_lineales["PCA_2_vs_control"] = (
    tabla_lineales["PCA_2"] - tabla_lineales["Control"]
)

tabla_lineales["PCA_3_vs_control"] = (
    tabla_lineales["PCA_3"] - tabla_lineales["Control"]
)

tabla_lineales["PCA_3_vs_PCA_2"] = (
    tabla_lineales["PCA_3"] - tabla_lineales["PCA_2"]
)

display(tabla_lineales.round(4))

print("\nPromedios por modelo:")
display(
    tabla_lineales[
        ["Control", "Local_3", "PCA_2", "PCA_3"]
    ].mean().round(4)
)

#### PCA cuadrático diagonal


In [ ]:
# PCA cuadrático diagonal con dos componentes

modelo_pca_2_cuadratico = clone(modelo_pca_2)

modelo_pca_2_cuadratico.set_params(
    preprocesamiento__pca=Pipeline([
        (
            "cuadrados",
            FunctionTransformer(
                agregar_cuadrados_cinematicos,
                validate=False
            )
        ),
        ("escalamiento", StandardScaler())
    ])
)

metricas_pca_2_cuadratico, predicciones_pca_2_cuadratico = (
    evaluar_modelo_temporal(
        modelo_pca_2_cuadratico,
        "Control + PCA cuadratico diagonal (2 PC)",
        df_entrenamiento_ml,
        pliegues_temporales_estandar,
        preparar_pliegue=preparar_pca_pliegue,
    )
)

# Comparar con el PCA lineal de dos componentes
comparacion_pca_cuadratico = (
    metricas_pca_2_cuadratico.loc[
        metricas_pca_2_cuadratico["muestra"] == "Estándar",
        ["anio_validacion", "roc_auc"]
    ]
    .rename(columns={"roc_auc": "PCA_2_cuadratico"})
    .merge(
        metricas_pca_2.loc[
            metricas_pca_2["muestra"] == "Estándar",
            ["anio_validacion", "roc_auc"]
        ].rename(columns={"roc_auc": "PCA_2_lineal"}),
        on="anio_validacion",
        validate="one_to_one"
    )
)

comparacion_pca_cuadratico["mejora_cuadratico"] = (
    comparacion_pca_cuadratico["PCA_2_cuadratico"]
    - comparacion_pca_cuadratico["PCA_2_lineal"]
)

display(comparacion_pca_cuadratico.round(4))

print(
    "Mejora media:",
    comparacion_pca_cuadratico["mejora_cuadratico"].mean().round(4)
)

print(
    "Años con mejora:",
    (comparacion_pca_cuadratico["mejora_cuadratico"] > 0).sum(),
    "de 5"
)

In [ ]:
# PCA cuadrático con tres componentes

preprocesador_pca_3_cuadratico = ColumnTransformer(
    [
        ("fase", clone(transformador_fase), variables_fase),
        ("calidad", clone(transformador_calidad), variables_calidad),
        (
            "pca",
            Pipeline([
                (
                    "cuadrados",
                    FunctionTransformer(
                        agregar_cuadrados_cinematicos,
                        validate=False
                    )
                ),
                ("escalamiento", StandardScaler())
            ]),
            ["PC1", "PC2", "PC3"]
        ),
    ],
    remainder="drop",
)

modelo_pca_3_cuadratico = Pipeline([
    ("preprocesamiento", preprocesador_pca_3_cuadratico),
    ("clasificador", clone(modelo_pca_3.named_steps["clasificador"]))
])

metricas_pca_3_cuadratico, predicciones_pca_3_cuadratico = (
    evaluar_modelo_temporal(
        modelo_pca_3_cuadratico,
        "Control + PCA cuadratico diagonal (3 PC)",
        df_entrenamiento_ml,
        pliegues_temporales_estandar,
        preparar_pliegue=preparar_pca_pliegue,
    )
)

comparacion_cuadraticos = (
    metricas_pca_2_cuadratico.query("muestra == 'Estándar'")
    [["anio_validacion", "roc_auc"]]
    .rename(columns={"roc_auc": "PCA_2"})
    .merge(
        metricas_pca_3_cuadratico.query("muestra == 'Estándar'")
        [["anio_validacion", "roc_auc"]]
        .rename(columns={"roc_auc": "PCA_3"}),
        on="anio_validacion",
        validate="one_to_one"
    )
)

comparacion_cuadraticos["diferencia"] = (
    comparacion_cuadraticos["PCA_3"]
    - comparacion_cuadraticos["PCA_2"]
)

display(comparacion_cuadraticos.round(4))
print("Diferencia media:", comparacion_cuadraticos["diferencia"].mean().round(4))

## 4. Comparación de los siete modelos


In [ ]:
# Comparar los siete modelos de clasificación

modelos = {
    "Control": metricas_control,
    "Local lineal": metricas_cinematica_lineal,
    "Local cuadrático": metricas_cinematica_cuadratica,
    "PCA 2 lineal": metricas_pca_2,
    "PCA 3 lineal": metricas_pca_3,
    "PCA 2 cuadrático": metricas_pca_2_cuadratico,
    "PCA 3 cuadrático": metricas_pca_3_cuadratico,
}

columnas = [
    "muestra", "anio_validacion", "trayectorias",
    "roc_auc", "average_precision", "exactitud_balanceada"
]

tabla_modelos = pd.concat(
    [
        metricas[columnas].assign(modelo=nombre)
        for nombre, metricas in modelos.items()
    ],
    ignore_index=True
)

# Comprobar que los modelos se comparan en los mismos años y tamaños
claves = ["muestra", "anio_validacion"]

assert not tabla_modelos.duplicated(claves + ["modelo"]).any()
assert (tabla_modelos.groupby(claves)["trayectorias"].nunique() == 1).all()
assert (tabla_modelos.groupby(claves)["modelo"].nunique() == 7).all()

# Añadir las métricas del control de cada año
control = tabla_modelos.loc[
    tabla_modelos["modelo"] == "Control",
    claves + ["roc_auc", "average_precision", "exactitud_balanceada"]
].rename(columns={
    "roc_auc": "auc_control",
    "average_precision": "ap_control",
    "exactitud_balanceada": "balanceada_control"
})

tabla_modelos = tabla_modelos.merge(
    control, on=claves, validate="many_to_one"
)

tabla_modelos["delta_auc"] = (
    tabla_modelos["roc_auc"] - tabla_modelos["auc_control"]
)

tabla_modelos["delta_ap"] = (
    tabla_modelos["average_precision"] - tabla_modelos["ap_control"]
)

tabla_modelos["delta_balanceada"] = (
    tabla_modelos["exactitud_balanceada"]
    - tabla_modelos["balanceada_control"]
)

# Desempeño promedio y mejoras respecto al control
resumen_modelos = tabla_modelos.groupby(
    ["muestra", "modelo"], as_index=False
).agg(
    auc_medio=("roc_auc", "mean"),
    ap_medio=("average_precision", "mean"),
    balanceada_media=("exactitud_balanceada", "mean"),
    mejora_auc=("delta_auc", "mean"),
    mejora_ap=("delta_ap", "mean"),
    anios_mejora=("delta_auc", lambda x: (x > 0).sum())
)

display(
    resumen_modelos.sort_values(
        ["muestra", "mejora_auc"],
        ascending=[True, False]
    ).round(4)
)

# Comparación anual de la muestra estándar
tabla_anual = tabla_modelos.loc[
    tabla_modelos["muestra"] == "Estándar"
].pivot(
    index="anio_validacion",
    columns="modelo",
    values="roc_auc"
)

display(tabla_anual.round(4))

## 5. Diagnósticos físicos y observacionales


In [ ]:
# Resumen de las componentes residuales por intervalos de fase solar

base_fase = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    ["delta_sol_lon_deg"] + variables_cinematica
].copy()

limites_fase = np.arange(-5, 2.5, 0.5)

base_fase["intervalo_fase"] = pd.cut(
    base_fase["delta_sol_lon_deg"],
    bins=limites_fase,
    right=False
)

fuera_ventana = base_fase["intervalo_fase"].isna().sum()
print("Trayectorias fuera de la ventana de fase:", fuera_ventana)

resumen_fase = base_fase.groupby(
    "intervalo_fase", observed=True
).agg(
    n=("delta_sol_lon_deg", "size"),
    media_paralela=(variables_cinematica[0], "mean"),
    sd_paralela=(variables_cinematica[0], "std"),
    media_lambda=(variables_cinematica[1], "mean"),
    sd_lambda=(variables_cinematica[1], "std"),
    media_beta=(variables_cinematica[2], "mean"),
    sd_beta=(variables_cinematica[2], "std")
)

display(resumen_fase.round(4))

### Dispersión por brillo y fase


In [ ]:
# Comparar la dispersión cinemática por brillo y fase solar

base_fase_brillo = df_entrenamiento.loc[
    df_entrenamiento["usar_modelo_ml"],
    ["delta_sol_lon_deg", "grupo_brillo_ml"] + variables_cinematica
].copy()

base_fase_brillo["intervalo_fase"] = pd.cut(
    base_fase_brillo["delta_sol_lon_deg"],
    bins=np.arange(-5, 2.5, 0.5),
    right=False
)

# Calcular el tamaño y la dispersión de cada grupo
tabla_grupos = (
    base_fase_brillo
    .groupby(["intervalo_fase", "grupo_brillo_ml"], observed=True)
    .agg(
        n=("grupo_brillo_ml", "size"),
        sd_paralela=(variables_cinematica[0], "std"),
        sd_lambda=(variables_cinematica[1], "std"),
        sd_beta=(variables_cinematica[2], "std")
    )
    .unstack("grupo_brillo_ml")
)

# Comparar la desviación estándar de tenues contra brillantes
tabla_dispersion_fase = pd.DataFrame(index=tabla_grupos.index)

for grupo in ["brillante", "tenue"]:
    tabla_dispersion_fase[f"n_{grupo}"] = (
        tabla_grupos[("n", grupo)].fillna(0).astype(int)
    )

for eje in ["paralela", "lambda", "beta"]:
    tabla_dispersion_fase[f"razon_{eje}"] = (
        tabla_grupos[(f"sd_{eje}", "tenue")]
        / tabla_grupos[(f"sd_{eje}", "brillante")]
    )

# Evitar interpretar intervalos con muy pocas observaciones
tabla_dispersion_fase["n_suficiente"] = (
    (tabla_dispersion_fase["n_brillante"] >= 50)
    & (tabla_dispersion_fase["n_tenue"] >= 50)
)

display(tabla_dispersion_fase.round(3))

print(
    "Intervalos con al menos 50 meteoros por grupo:",
    tabla_dispersion_fase["n_suficiente"].sum(),
    "de",
    len(tabla_dispersion_fase)
)

### Calidad observacional


In [ ]:
# Calidad observacional de brillantes y tenues

calidad_por_brillo = (
    df_entrenamiento_ml
    .groupby("grupo_brillo_ml", observed=True)[variables_calidad]
    .median()
    .T
)

display(calidad_por_brillo.round(4))

# Cuántos meteoros de cada grupo conserva la muestra estricta
retencion_calidad = (
    df_entrenamiento_ml
    .groupby("grupo_brillo_ml", observed=True)["pertenece_muestra_estricta"]
    .agg(n="size", estrictas="sum")
)

retencion_calidad["porcentaje_estricto"] = (
    100 * retencion_calidad["estrictas"] / retencion_calidad["n"]
)

display(retencion_calidad.round(2))

In [ ]:
# Comparación por fase utilizando solamente la muestra estricta

base_estricta_fase = df_entrenamiento_ml_estricto.copy()

base_estricta_fase["intervalo_fase"] = pd.cut(
    base_estricta_fase["delta_sol_lon_deg"],
    bins=np.arange(-5, 2.5, 0.5),
    right=False
)

resumen_estricto = (
    base_estricta_fase
    .groupby(
        ["intervalo_fase", "grupo_brillo_ml"],
        observed=True
    )[variables_cinematica[0]]
    .agg(n="size", sd="std")
    .unstack("grupo_brillo_ml")
)

comparacion_calidad_fase = (
    tabla_dispersion_fase[
        ["n_brillante", "n_tenue", "razon_paralela"]
    ]
    .rename(columns={"razon_paralela": "razon_estandar"})
    .copy()
)

for grupo in ["brillante", "tenue"]:
    comparacion_calidad_fase[f"n_estricto_{grupo}"] = (
        resumen_estricto[("n", grupo)]
        .reindex(comparacion_calidad_fase.index)
        .fillna(0)
        .astype(int)
    )

comparacion_calidad_fase["razon_estricta"] = (
    resumen_estricto[("sd", "tenue")]
    / resumen_estricto[("sd", "brillante")]
)

# Un mínimo exploratorio de 30 observaciones por clase
comparacion_calidad_fase["revisar"] = (
    (comparacion_calidad_fase["n_estricto_brillante"] >= 30)
    & (comparacion_calidad_fase["n_estricto_tenue"] >= 30)
)

display(comparacion_calidad_fase.round(3))

intervalos_revisables = comparacion_calidad_fase.loc[
    comparacion_calidad_fase["revisar"]
]

print("Intervalos revisables:", len(intervalos_revisables))

if len(intervalos_revisables) > 0:
    print(
        "Razón estricta > 1:",
        (intervalos_revisables["razon_estricta"] > 1).sum(),
        "de",
        len(intervalos_revisables)
    )

## 6. Auditorías de las predicciones y convergencia


In [ ]:
# Comprobar que todos los modelos evaluaron las mismas trayectorias

predicciones_modelos = {
    "Control": predicciones_control,
    "Local lineal": predicciones_cinematica_lineal,
    "Local cuadrático": predicciones_cinematica_cuadratica,
    "PCA 2 lineal": predicciones_pca_2,
    "PCA 3 lineal": predicciones_pca_3,
    "PCA 2 cuadrático": predicciones_pca_2_cuadratico,
    "PCA 3 cuadrático": predicciones_pca_3_cuadratico,
}

columnas_revision = [
    "id_fila_origen",
    "anio_validacion",
    "objetivo_real",
    "pertenece_muestra_estricta"
]

referencia = (
    predicciones_control[columnas_revision]
    .sort_values(["anio_validacion", "id_fila_origen"])
    .reset_index(drop=True)
)

filas_revision = []

for nombre, predicciones in predicciones_modelos.items():

    datos = (
        predicciones[columnas_revision]
        .sort_values(["anio_validacion", "id_fila_origen"])
        .reset_index(drop=True)
    )

    mismas_filas = datos.equals(referencia)

    probabilidades = predicciones["probabilidad_tenue"]
    probabilidades_validas = (
        probabilidades.between(0, 1).all()
        and np.isfinite(probabilidades).all()
    )

    ids_unicos = predicciones["id_fila_origen"].is_unique

    filas_revision.append({
        "modelo": nombre,
        "mismas_filas": mismas_filas,
        "ids_unicos": ids_unicos,
        "probabilidades_validas": probabilidades_validas
    })

auditoria_predicciones = pd.DataFrame(filas_revision)

display(auditoria_predicciones)

assert auditoria_predicciones.iloc[:, 1:].to_numpy().all()

In [ ]:
# Comprobar convergencia en los cinco pliegues temporales

modelos_auditar = {
    "Control": (modelo_control, None),
    "Local lineal": (modelo_cinematica_lineal, None),
    "Local cuadrático": (modelo_cinematica_cuadratica, None),
    "PCA 2 lineal": (modelo_pca_2, preparar_pca_pliegue),
    "PCA 3 lineal": (modelo_pca_3, preparar_pca_pliegue),
    "PCA 2 cuadrático": (
        modelo_pca_2_cuadratico, preparar_pca_pliegue
    ),
    "PCA 3 cuadrático": (
        modelo_pca_3_cuadratico, preparar_pca_pliegue
    ),
}

revision_convergencia = []

for nombre, (modelo, preparar) in modelos_auditar.items():

    for indices_train, indices_val in pliegues_temporales_estandar:

        X_train = df_entrenamiento_ml.iloc[indices_train].copy()
        X_val = df_entrenamiento_ml.iloc[indices_val].copy()

        anio = int(X_val["source_year"].iloc[0])
        y_train = X_train["objetivo_ml"].astype(int)

        if preparar is not None:
            X_train, X_val = preparar(X_train, X_val, anio)

        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always", ConvergenceWarning)

            ajuste = clone(modelo)
            ajuste.fit(X_train, y_train)

        advertencia = any(
            issubclass(aviso.category, ConvergenceWarning)
            for aviso in avisos
        )

        iteraciones = int(
            ajuste.named_steps["clasificador"].n_iter_.max()
        )

        revision_convergencia.append({
            "modelo": nombre,
            "anio": anio,
            "iteraciones": iteraciones,
            "advertencia": advertencia
        })

tabla_convergencia = pd.DataFrame(revision_convergencia)

display(
    tabla_convergencia.groupby("modelo").agg(
        max_iteraciones=("iteraciones", "max"),
        advertencias=("advertencia", "sum")
    )
)

print(
    "Ajustes con advertencia:",
    tabla_convergencia["advertencia"].sum(),
    "de",
    len(tabla_convergencia)
)


## 7. Selección y protocolo pre-2025


In [ ]:
# Modelos que consideraremos para la evaluación final

candidatos_finales = [
    "Control",
    "Local cuadrático",
    "PCA 2 cuadrático"
]

tabla_seleccion = resumen_modelos.loc[
    (resumen_modelos["muestra"] == "Estándar")
    & (resumen_modelos["modelo"].isin(candidatos_finales)),
    [
        "modelo",
        "auc_medio",
        "ap_medio",
        "balanceada_media",
        "mejora_auc",
        "anios_mejora"
    ]
].sort_values("auc_medio", ascending=False)

assert len(tabla_seleccion) == 3
display(tabla_seleccion.round(4))

### Protocolo congelado


In [ ]:
# Registrar las decisiones tomadas con 2019–2024

clasificador_final = (
    modelo_cinematica_cuadratica
    .named_steps["clasificador"]
)

protocolo_ml = {
    "version": "pre2025_v1",
    "entrenamiento": "2019-2024",
    "prueba_reservada": 2025,
    "muestra_principal": "estandar",
    "objetivo": "objetivo_ml",
    "clase_positiva": "tenue",
    "limite_brillante": float(limite_brillante),
    "limite_tenue": float(limite_tenue),

    "modelo_principal": "local_cuadratico_diagonal",
    "modelo_control": "control",
    "modelo_alternativo": "pca_2_cuadratico_diagonal",

    "variables_fase": variables_fase,
    "grado_fase": 2,
    "variables_calidad": variables_calidad,
    "variables_cinematica": variables_cinematica,

    "regresion": {
        "C": float(clasificador_final.C),
        "solver": clasificador_final.solver,
        "max_iter": int(clasificador_final.max_iter),
    },

    "umbral_clasificacion": 0.5,
    "metrica_principal": "roc_auc",
    "metricas_secundarias": [
        "average_precision",
        "exactitud_balanceada",
    ],

    "validacion_interna": {
        "tipo": "temporal_creciente_condicional",
        "anios_validacion": [2020, 2021, 2022, 2023, 2024],
        "centro_fisico": "fijado en notebook 04 con 2019-2024",
        "limites_clase": "fijados en notebook 04 con 2019-2024",
        "ajuste_por_pliegue": (
            "StandardScaler, PCA cuando corresponde y regresion "
            "logistica ajustados solo con anos anteriores"
        ),
        "nota": (
            "No es una validacion end-to-end del centro fisico "
            "ni de la generacion de etiquetas."
        ),
    },

    "evaluacion_estricta": "subconjunto_del_modelo_estandar",
    "ajustar_pca_con": (
        "todas_las_trayectorias_elegibles_del_entrenamiento"
    ),
    "numero_pc_alternativa": 2,
    "reajustar_con_2025": False,
}

ruta_config = (
    raiz_proyecto
    / "configs"
    / "protocolo_ml_pre2025.json"
)

ruta_config.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with ruta_config.open("w", encoding="utf-8") as archivo:
    json.dump(
        protocolo_ml,
        archivo,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Protocolo guardado:",
    ruta_config.relative_to(raiz_proyecto),
)


## 8. Estabilidad del resultado


In [ ]:
# Diferencias por año entre los modelos seleccionados

metrica_ap = ("average_precision")

metricas_revisar = ["roc_auc", metrica_ap, "exactitud_balanceada"]

referencias = {
    "Control": metricas_control,
    "PCA 2 cuadrático": metricas_pca_2_cuadratico,
    "PCA 3 cuadrático": metricas_pca_3_cuadratico,
}

principal = metricas_cinematica_cuadratica.query(
    "muestra == 'Estándar'"
)

filas_comparacion = []

for nombre, tabla in referencias.items():
    referencia = tabla.query("muestra == 'Estándar'")

    comparacion = principal.merge(
        referencia,
        on="anio_validacion",
        suffixes=("_local", "_referencia"),
        validate="one_to_one"
    )

    for _, fila in comparacion.iterrows():
        filas_comparacion.append({
            "comparacion": f"Local - {nombre}",
            "anio": int(fila["anio_validacion"]),
            "delta_auc": (
                fila["roc_auc_local"] - fila["roc_auc_referencia"]
            ),
            "delta_ap": (
                fila[f"{metrica_ap}_local"]
                - fila[f"{metrica_ap}_referencia"]
            ),
            "delta_balanceada": (
                fila["exactitud_balanceada_local"]
                - fila["exactitud_balanceada_referencia"]
            )
        })

tabla_diferencias = pd.DataFrame(filas_comparacion)

display(tabla_diferencias.round(4))

display(
    tabla_diferencias.groupby("comparacion")[
        ["delta_auc", "delta_ap", "delta_balanceada"]
    ].mean().round(4)
)

In [ ]:
# Coeficientes del modelo principal en cada pliegue temporal

nombres_terminos = (
    ["fase", "fase_cuadrada"]
    + variables_calidad
    + variables_cinematica
    + [f"{v}^2" for v in variables_cinematica]
)

filas_coeficientes = []

for indices_train, indices_val in pliegues_temporales_estandar:

    datos_train = df_entrenamiento_ml.iloc[indices_train]
    datos_val = df_entrenamiento_ml.iloc[indices_val]

    anio = int(datos_val["source_year"].iloc[0])

    ajuste = clone(modelo_cinematica_cuadratica)

    ajuste.fit(
        datos_train,
        datos_train["objetivo_ml"].astype(int)
    )

    coeficientes = ajuste.named_steps["clasificador"].coef_[0]

    assert len(coeficientes) == len(nombres_terminos)

    for termino, coeficiente in zip(nombres_terminos, coeficientes):
        filas_coeficientes.append({
            "anio_validacion": anio,
            "termino": termino,
            "coeficiente": coeficiente
        })

tabla_coeficientes = pd.DataFrame(filas_coeficientes)

terminos_cinematica = (
    variables_cinematica
    + [f"{v}^2" for v in variables_cinematica]
)

tabla_estabilidad = tabla_coeficientes.pivot(
    index="anio_validacion",
    columns="termino",
    values="coeficiente"
)[terminos_cinematica]

display(tabla_estabilidad.round(4))

termino_paralelo_cuadrado = f"{variables_cinematica[0]}^2"

print(
    "Signos del coeficiente paralelo cuadrático:",
    np.sign(tabla_estabilidad[termino_paralelo_cuadrado]).tolist()
)

In [ ]:
# Revisar las direcciones PCA aprendidas en cada pliegue

datos_globales = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    variables_cinematica
]

pca_referencia = PCA(n_components=3).fit(datos_globales)

filas_estabilidad_pca = []

for anio in range(2020, 2025):

    datos_anteriores = df_entrenamiento.loc[
        df_entrenamiento["usar_pca"]
        & (df_entrenamiento["source_year"] < anio),
        variables_cinematica
    ]

    pca_anio = PCA(n_components=3).fit(datos_anteriores)

    # Comparar direcciones sin depender del signo arbitrario
    similitud = np.abs(
        pca_anio.components_ @ pca_referencia.components_.T
    )

    filas_estabilidad_pca.append({
        "anio_validacion": anio,
        "n_ajuste": len(datos_anteriores),
        "PC1_var_%": 100 * pca_anio.explained_variance_ratio_[0],
        "PC2_var_%": 100 * pca_anio.explained_variance_ratio_[1],
        "PC3_var_%": 100 * pca_anio.explained_variance_ratio_[2],
        "similitud_PC1": similitud[0, 0],
        "PC2_mas_parecida_a": f"PC{similitud[1].argmax() + 1}",
        "PC3_mas_parecida_a": f"PC{similitud[2].argmax() + 1}"
    })

tabla_estabilidad_pca = pd.DataFrame(filas_estabilidad_pca)

display(tabla_estabilidad_pca.round(4))


### Sensibilidad a extremos fotométricos


In [ ]:
# Corte del 1 % más brillante, definido en entrenamiento

magnitudes_brillantes = df_entrenamiento_ml.loc[
    df_entrenamiento_ml["objetivo_ml"] == 0,
    "magnitud_analisis"
]

corte_extremos = float(magnitudes_brillantes.quantile(0.01))

n_excluidos = (magnitudes_brillantes < corte_extremos).sum()

print("Corte de magnitud:", round(corte_extremos, 4))
print("Brillantes excluidos:", n_excluidos)
print("Brillantes originales:", len(magnitudes_brillantes))

In [ ]:
# Evaluar el control y el modelo principal sin los extremos brillantes

predicciones_revisar = {
    "Control": predicciones_control,
    "Local cuadrático": predicciones_cinematica_cuadratica
}

magnitudes_por_id = df_entrenamiento_ml[
    ["id_fila_origen", "magnitud_analisis"]
]

filas_sensibilidad = []

for nombre, predicciones in predicciones_revisar.items():

    datos = predicciones.merge(
        magnitudes_por_id,
        on="id_fila_origen",
        validate="one_to_one"
    )

    for anio, grupo in datos.groupby("anio_validacion"):

        conjuntos = {
            "Original": grupo,
            "Sin extremos": grupo.loc[
                grupo["magnitud_analisis"] >= corte_extremos
            ]
        }

        for seleccion, muestra in conjuntos.items():

            y = muestra["objetivo_real"].astype(int)
            p = muestra["probabilidad_tenue"]
            pred = muestra["prediccion"].astype(int)

            assert y.nunique() == 2

            filas_sensibilidad.append({
                "modelo": nombre,
                "anio": anio,
                "seleccion": seleccion,
                "n": len(muestra),
                "roc_auc": roc_auc_score(y, p),
                "average_precision": average_precision_score(y, p),
                "exactitud_balanceada": balanced_accuracy_score(y, pred)
            })

tabla_sensibilidad = pd.DataFrame(filas_sensibilidad)

local = tabla_sensibilidad.query(
    "modelo == 'Local cuadrático'"
)

control = tabla_sensibilidad.query(
    "modelo == 'Control'"
)

comparacion_extremos = local.merge(
    control,
    on=["anio", "seleccion"],
    suffixes=("_local", "_control"),
    validate="one_to_one"
)

for metrica in [
    "roc_auc", "average_precision", "exactitud_balanceada"
]:
    comparacion_extremos[f"mejora_{metrica}"] = (
        comparacion_extremos[f"{metrica}_local"]
        - comparacion_extremos[f"{metrica}_control"]
    )

display(
    comparacion_extremos[
        [
            "anio", "seleccion", "n_local",
            "mejora_roc_auc",
            "mejora_average_precision",
            "mejora_exactitud_balanceada"
        ]
    ].round(4)
)

## 9. Ajuste final con 2019–2024


In [ ]:
# Ajustar los modelos finales utilizando todo el entrenamiento

from joblib import dump, load

assert df_entrenamiento_ml["source_year"].between(2019, 2024).all()

y_final = df_entrenamiento_ml["objetivo_ml"].astype(int)

# Control y modelo local cuadrático
control_final = clone(modelo_control)
control_final.fit(df_entrenamiento_ml, y_final)

local_final = clone(modelo_cinematica_cuadratica)
local_final.fit(df_entrenamiento_ml, y_final)

# PCA global: todos los meteoros elegibles de entrenamiento
datos_pca_final = df_entrenamiento.loc[
    df_entrenamiento["usar_pca"],
    variables_cinematica
]

pca_final = PCA(n_components=3)
pca_final.fit(datos_pca_final)

# Proyectar las filas utilizadas por la regresión
datos_ml_pca = df_entrenamiento_ml.copy()
coordenadas = pca_final.transform(
    datos_ml_pca[variables_cinematica]
)

datos_ml_pca["PC1"] = coordenadas[:, 0]
datos_ml_pca["PC2"] = coordenadas[:, 1]

# Modelo reducido con PCA
pca2_final = clone(modelo_pca_2_cuadratico)
pca2_final.fit(datos_ml_pca, y_final)

print("Meteoros para regresión:", len(df_entrenamiento_ml))
print("Meteoros para PCA:", len(datos_pca_final))

for nombre, modelo in [
    ("Control", control_final),
    ("Local cuadrático", local_final),
    ("PCA 2 cuadrático", pca2_final),
]:
    print(
        nombre,
        "- iteraciones:",
        modelo.named_steps["clasificador"].n_iter_[0]
    )


In [ ]:
# Guardar modelos y transformaciones aprendidas

ruta_modelos = (
    raiz_proyecto / "reports" / "models"
    / "modelos_pre2025.joblib"
)

ruta_modelos.parent.mkdir(parents=True, exist_ok=True)

paquete_modelos = {
    "control": control_final,
    "local_cuadratico": local_final,
    "pca_2_cuadratico": pca2_final,
    "pca_global": pca_final,
    "variables_cinematica": variables_cinematica,
}

dump(paquete_modelos, ruta_modelos)

# Comprobar que el archivo conserva las predicciones
modelos_recuperados = load(ruta_modelos)

for nombre, datos in [
    ("control", df_entrenamiento_ml),
    ("local_cuadratico", df_entrenamiento_ml),
    ("pca_2_cuadratico", datos_ml_pca),
]:
    probabilidades_originales = (
        paquete_modelos[nombre].predict_proba(datos)[:, 1]
    )

    probabilidades_recuperadas = (
        modelos_recuperados[nombre].predict_proba(datos)[:, 1]
    )

    np.testing.assert_allclose(
        probabilidades_originales,
        probabilidades_recuperadas,
        rtol=0,
        atol=1e-12
    )

print("Modelos guardados:", ruta_modelos)
print("Comprobación de recuperación: correcta")

In [ ]:
# Completar el protocolo con las decisiones finales

protocolo_ml["sensibilidad_fotometrica"] = {
    "tipo": "exclusion_de_extremos_en_evaluacion",
    "variable": "magnitud_analisis",
    "percentil": 0.01,
    "poblacion_para_corte": "brillantes_2019_2024",
    "corte": float(corte_extremos),
    "excluidos_en_desarrollo": int(n_excluidos),
    "reentrenamiento": False,
}

protocolo_ml["filas_entrenamiento_ml"] = int(len(df_entrenamiento_ml))
protocolo_ml["filas_ajuste_pca"] = int(len(datos_pca_final))

protocolo_ml["transformacion_cinematica"] = (
    "componentes locales, cuadrados y StandardScaler"
)

protocolo_ml["transformacion_pca_alternativa"] = (
    "PCA ajustado con todas las trayectorias elegibles "
    "de 2019-2024; utilizar PC1 y PC2"
)

protocolo_ml["archivo_modelos"] = (
    ruta_modelos
    .relative_to(raiz_proyecto)
    .as_posix()
)

protocolo_ml["sha256_modelos"] = hashlib.sha256(
    ruta_modelos.read_bytes()
).hexdigest()

with open(ruta_config, "w", encoding="utf-8") as archivo:
    json.dump(protocolo_ml, archivo, indent=2, ensure_ascii=False)

print("Protocolo actualizado:", ruta_config)
print("Corte de magnitud:", protocolo_ml["sensibilidad_fotometrica"]["corte"])
print("SHA-256:", protocolo_ml["sha256_modelos"])


## 10. Exportación de resultados de desarrollo


In [ ]:
# Guardar las tablas necesarias para reconstruir la selección de modelos

ruta_tablas = raiz_proyecto / "reports" / "tables"
ruta_tablas.mkdir(parents=True, exist_ok=True)

tabla_modelos.to_csv(
    ruta_tablas / "ml_validacion_interna_metricas_2019_2024.csv",
    index=False,
)

predicciones_internas = pd.concat(
    [
        tabla.assign(modelo=nombre)
        for nombre, tabla in predicciones_modelos.items()
    ],
    ignore_index=True,
)

predicciones_internas.to_parquet(
    ruta_tablas / "ml_validacion_interna_predicciones_2019_2024.parquet",
    index=False,
)

tabla_coeficientes.to_csv(
    ruta_tablas / "ml_estabilidad_coeficientes_2019_2024.csv",
    index=False,
)

tabla_estabilidad_pca.to_csv(
    ruta_tablas / "ml_estabilidad_pca_2019_2024.csv",
    index=False,
)

comparacion_extremos.to_csv(
    ruta_tablas / "ml_sensibilidad_extremos_2019_2024.csv",
    index=False,
)

tabla_dispersion_fase.reset_index().to_csv(
    ruta_tablas / "ml_dispersion_fase_2019_2024.csv",
    index=False,
)

print("Resultados internos guardados en reports/tables/.")
